# Raw vs Curated — operational min-8 audit V5 evaluation

This notebook evaluates **12 fixed checkpoints** (Raw/Curated × six architectures) using one unified evaluator on exactly the same held-out test:

```text
benchmakring_transmission_line_defect_model/data/test_operational_min8_v5_full_consensus_assisted/
├── images/test   # 1,871 images
└── labels/test   # 1,871 label files, 2,467 boxes
```

This is the **model-assisted exploratory V5 operational subset**, not the canonical independent `test_main_v1`. Report it explicitly as V5 audit results.

Expected hardware: RTX 3050/3060. Checkpoints run sequentially to fit limited VRAM. Existing per-run JSON files are reused, so the notebook can safely resume after interruption.


In [ ]:
# 1. Resolve repository and evaluation paths.
from pathlib import Path
import os, sys

def find_repo(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'benchmakring_transmission_line_defect_model').is_dir():
            return p
    raise FileNotFoundError('Open this notebook from inside the yolov11 repository.')

REPO = find_repo(Path.cwd())
BENCH = REPO / 'benchmakring_transmission_line_defect_model'
DATASET_DIR = BENCH / 'data' / 'test_operational_min8_v5_full_consensus_assisted'
DATASET_YAML = DATASET_DIR / 'dataset.yaml'
MODEL_ROOT = BENCH / 'artifacts' / 'raw_curated_fixed_checkpoints'
OUTPUT_DIR = BENCH / 'results_ablation' / 'finding_nemo' / 'raw_curated_operational_min8_v5'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Repository :', REPO)
print('Dataset    :', DATASET_YAML)
print('Models     :', MODEL_ROOT)
print('Output     :', OUTPUT_DIR)


## Environment

Install CUDA-enabled PyTorch **before** running this notebook. The cell below deliberately does not reinstall PyTorch because doing so can replace a working CUDA build with a CPU build.


In [ ]:
# 2. Install evaluator dependencies. Restart the kernel if pip asks for it.
%pip install -q "ultralytics==8.4.172" "transformers==4.47.1" "accelerate>=1.0,<2" pycocotools pandas matplotlib pyyaml pillow


In [ ]:
# 3. Hardware fail-fast check and conservative automatic batches.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. Install the CUDA build of PyTorch and restart the kernel; '
        'do not run the 12-model benchmark on CPU.'
    )

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 2**30
DEVICE = '0'
YOLO_BATCH = 16 if vram_gb >= 10 else (8 if vram_gb >= 6 else 4)
RTDETR_BATCH = 4 if vram_gb >= 10 else (2 if vram_gb >= 7 else 1)
print(f'GPU: {gpu.name} | VRAM: {vram_gb:.1f} GiB')
print(f'YOLO batch={YOLO_BATCH}; RT-DETR batch={RTDETR_BATCH}')


In [ ]:
# 4. Verify the operational min-8 audit V5 test before spending GPU time.
image_dir = DATASET_DIR / 'images' / 'test'
label_dir = DATASET_DIR / 'labels' / 'test'
images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})
labels = sorted(label_dir.glob('*.txt'))
boxes = sum(sum(bool(line.strip()) for line in p.read_text(encoding='utf-8', errors='ignore').splitlines()) for p in labels)
missing = [p.stem for p in images if not (label_dir / f'{p.stem}.txt').is_file()]

print(f'images={len(images):,}; labels={len(labels):,}; boxes={boxes:,}; missing_labels={len(missing)}')
assert DATASET_YAML.is_file()
assert len(images) == 1871, 'Expected exactly 1,871 operational V5 test images.'
assert len(labels) == 1871 and not missing, 'Image/label pairing is not frozen as expected.'
assert boxes == 2467, 'Expected exactly 2,467 operational V5 GT boxes.'
print('OPERATIONAL MIN-8 AUDIT V5 VERIFIED')


In [ ]:
# 5. Fixed Raw/Curated checkpoint manifest + robust local/Camber restore.
import shutil, subprocess, shlex

CHECKPOINTS = {
    ('Raw','yolo11n'): MODEL_ROOT/'raw/yolo11n/weights/best.pt',
    ('Raw','yolo11m'): MODEL_ROOT/'raw/yolo11m/weights/best.pt',
    ('Raw','yolo26n'): MODEL_ROOT/'raw/yolo26n/weights/best.pt',
    ('Raw','yolo26m'): MODEL_ROOT/'raw/yolo26m/weights/best.pt',
    ('Raw','rtdetr50'): MODEL_ROOT/'raw/rtdetr50/best.pt',
    ('Raw','rtdetr101'): MODEL_ROOT/'raw/rtdetr101/best.pt',
    ('Curated','yolo11n'): MODEL_ROOT/'curated/yolo11n/weights/best.pt',
    ('Curated','yolo11m'): MODEL_ROOT/'curated/yolo11m/weights/best.pt',
    ('Curated','yolo26n'): MODEL_ROOT/'curated/yolo26n/weights/best.pt',
    ('Curated','yolo26m'): MODEL_ROOT/'curated/yolo26m/weights/best.pt',
    ('Curated','rtdetr50'): MODEL_ROOT/'curated/rtdetr50/best.pt',
    ('Curated','rtdetr101'): MODEL_ROOT/'curated/rtdetr101/best.pt',
}

# Six curated checkpoints already downloaded by the selected-checkpoint benchmark.
LOCAL_ALIASES = {
    ('Curated','yolo11n'): BENCH/'artifacts/finding_nemo_selected_checkpoints/yolo11n_last.pt',
    ('Curated','yolo11m'): BENCH/'artifacts/finding_nemo_selected_checkpoints/yolo11m_epoch45.pt',
    ('Curated','yolo26n'): BENCH/'artifacts/finding_nemo_selected_checkpoints/yolo26n_epoch45.pt',
    ('Curated','yolo26m'): BENCH/'artifacts/finding_nemo_selected_checkpoints/yolo26m_epoch45.pt',
    ('Curated','rtdetr50'): BENCH/'artifacts/finding_nemo_selected_checkpoints/rtdetr50_epoch15.pt',
    ('Curated','rtdetr101'): BENCH/'artifacts/finding_nemo_selected_checkpoints/rtdetr101_epoch15.pt',
}

STASH = {
    ('Raw','yolo11n'): 'stash://bunpmc/projects/raw_yolo11n/weights/best.pt',
    ('Raw','yolo11m'): 'stash://bunpmc/projects/raw_yolo11m/weights/best.pt',
    ('Raw','yolo26n'): 'stash://bunpmc/projects/raw_yolo26n/weights/best.pt',
    ('Raw','yolo26m'): 'stash://bunpmc/projects/raw_yolo26m/weights/best.pt',
    ('Raw','rtdetr50'): 'stash://bunpmc/projects/raw_rtdetr50/checkpoints/best.pt',
    ('Raw','rtdetr101'): 'stash://bunpmc/projects/raw_rtdetr101/checkpoints/best.pt',
    ('Curated','yolo11n'): 'stash://bunpmc/projects/finding_nemo/yolo11n/weights/last.pt',
    ('Curated','yolo11m'): 'stash://bunpmc/projects/finding_nemo/yolo11m/weights/epoch45.pt',
    ('Curated','yolo26n'): 'stash://bunpmc/projects/finding_nemo/yolo26n/weights/epoch45.pt',
    ('Curated','yolo26m'): 'stash://bunpmc/projects/finding_nemo/yolo26m/weights/epoch45.pt',
    ('Curated','rtdetr50'): 'stash://bunpmc/projects/finding_nemo/rtdetr_training/finding_nemo_human_reviewed_6cls_20261101/r50vd/checkpoints/epoch15.pt',
    ('Curated','rtdetr101'): 'stash://bunpmc/projects/finding_nemo/rtdetr_training/finding_nemo_human_reviewed_6cls_20261101/r101vd/checkpoints/epoch15.pt',
}

# Mirror aliases into a stable manifest without duplicating hundreds of MB when hardlinks work.
for key, source in LOCAL_ALIASES.items():
    destination = CHECKPOINTS[key]
    if destination.is_file() or not source.is_file():
        continue
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        destination.hardlink_to(source)
    except OSError:
        shutil.copy2(source, destination)
    print(f'[LOCAL] {key} <- {source.relative_to(REPO)}')

missing_keys = [key for key, path in CHECKPOINTS.items() if not path.is_file()]

# API key may live in the repository .env. Never print it.
if not os.environ.get('CAMBER_API_KEY') and (REPO / '.env').is_file():
    for line in (REPO / '.env').read_text(encoding='utf-8-sig').splitlines():
        if line.strip().startswith('CAMBER_API_KEY='):
            os.environ['CAMBER_API_KEY'] = line.split('=', 1)[1].strip().strip('"').strip("'")
            break

CAMBER = shutil.which('camber') or shutil.which('camber.exe')
WSL = shutil.which('wsl') or shutil.which('wsl.exe')
WSL_CAMBER = False
if not CAMBER and WSL:
    probe = subprocess.run([WSL, 'sh', '-lc', '~/.camber/bin/camber version'],
                           capture_output=True, text=True)
    WSL_CAMBER = probe.returncode == 0

def camber_run(parts, destination=None):
    """Run native Camber, or the official Linux CLI through WSL on Windows."""
    if CAMBER:
        return subprocess.run([CAMBER, *parts], capture_output=True, text=True)
    if not WSL_CAMBER:
        raise RuntimeError('Không tìm thấy Camber CLI native hoặc ~/.camber/bin/camber trong WSL.')
    env = os.environ.copy()
    env['WSLENV'] = ':'.join(x for x in [env.get('WSLENV', ''), 'CAMBER_API_KEY'] if x)
    if destination is None:
        command = '~/.camber/bin/camber ' + ' '.join(parts)
        return subprocess.run([WSL, 'sh', '-lc', command], capture_output=True, text=True, env=env)
    # Convert C:\\path -> /mnt/c/path in Python. This avoids the multiple
    # PowerShell/wsl.exe/sh quoting layers that can eat every backslash.
    win_path = str(destination.resolve())
    if len(win_path) < 3 or win_path[1:3] != ':\\':
        raise RuntimeError(f'Không chuyển được đường dẫn Windows sang WSL: {win_path}')
    wsl_destination = '/mnt/' + win_path[0].lower() + win_path[2:].replace('\\', '/')
    # wsl.exe can drop positional args following `sh -lc` on some Windows
    # configurations, so quote both values into the Linux command itself.
    command = ('~/.camber/bin/camber stash cp '
               + shlex.quote(parts[-2]) + ' ' + shlex.quote(wsl_destination))
    return subprocess.run([WSL, 'sh', '-lc', command],
                          capture_output=True, text=True, env=env)

if missing_keys and (CAMBER or WSL_CAMBER):
    if not os.environ.get('CAMBER_API_KEY'):
        raise RuntimeError('Camber CLI đã có nhưng thiếu CAMBER_API_KEY trong environment hoặc file .env.')
    print('Camber CLI:', CAMBER or 'WSL ~/.camber/bin/camber')
    auth = camber_run(['me'])
    if auth.returncode:
        raise RuntimeError('Camber CLI có nhưng chưa xác thực: ' + (auth.stderr or auth.stdout)[-1000:])
    for key in missing_keys:
        destination = CHECKPOINTS[key]
        destination.parent.mkdir(parents=True, exist_ok=True)
        print(f'[CAMBER] {key} -> {destination.relative_to(REPO)}')
        result = camber_run(['stash', 'cp', STASH[key], str(destination)], destination=destination)
        if result.returncode:
            raise RuntimeError(f'Camber restore lỗi cho {key}:\n{(result.stderr or result.stdout)[-1500:]}')

missing_keys = [key for key, path in CHECKPOINTS.items() if not path.is_file()]
if missing_keys:
    details = '\n'.join(f'  - {key}: {STASH[key]}' for key in missing_keys)
    raise RuntimeError(
        'Thiếu checkpoint và không tìm thấy Camber CLI native hoặc Camber CLI trong WSL.\n'
        'Cài/đăng nhập Camber CLI rồi chạy lại cell này, hoặc chép các file sau thủ công:\n' + details
    )

for (regime, model), path in CHECKPOINTS.items():
    print(f'{regime:7} {model:10} {path.stat().st_size/2**20:7.1f} MiB  {path.relative_to(REPO)}')


If the local model mirror is absent, restore the same paths using `camber stash cp`. The source prefixes are:

```text
Raw YOLO:       stash://bunpmc/projects/raw_<model>/weights/best.pt
Raw RT-DETR:    stash://bunpmc/projects/raw_rtdetr50|101/checkpoints/best.pt
Curated YOLO:   stash://bunpmc/projects/finding_nemo/<model>/weights/best.pt
Curated RT-DETR stash://bunpmc/projects/finding_nemo/rtdetr_training/finding_nemo_human_reviewed_6cls_20261101/<r50vd|r101vd>/checkpoints/best.pt
```


In [ ]:
# 6. Load the repository's unified evaluator.
import gc, importlib.util, json, time
import pandas as pd

module_path = BENCH / 'scripts' / 'evaluate_nemo_selected_checkpoints.py'
spec = importlib.util.spec_from_file_location('nemo_eval', module_path)
nemo_eval = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nemo_eval)

MODEL_ORDER = ['yolo11n','yolo11m','yolo26n','yolo26m','rtdetr50','rtdetr101']
CLASSES = list(nemo_eval.CLASS_NAMES.values())
print('Unified evaluator loaded:', module_path)


In [ ]:
# 7. Evaluate sequentially. Re-run this cell to resume; completed JSON files are reused.
all_results = {}
for regime in ('Raw','Curated'):
    for model in MODEL_ORDER:
        checkpoint = CHECKPOINTS[(regime, model)]
        result_file = OUTPUT_DIR / f'{regime.lower()}_{model}.json'
        if result_file.is_file():
            result = json.loads(result_file.read_text(encoding='utf-8'))
            print(f'[REUSE] {regime:7} {model:10} mAP50={result["accuracy"]["mAP50"]:.4f}')
        else:
            print(f'\n[EVAL] {regime:7} {model:10} -> operational min-8 audit V5')
            if model.startswith('yolo'):
                result = nemo_eval.evaluate_yolo_checkpoint(
                    model_key=model, checkpoint_path=checkpoint, dataset_yaml=DATASET_YAML,
                    split='test', device_str=DEVICE, batch_size=YOLO_BATCH, limit=None, imgsz=640)
            else:
                result = nemo_eval.evaluate_rtdetr_checkpoint(
                    model_key=model, checkpoint_path=checkpoint, dataset_yaml=DATASET_YAML,
                    split='test', device_str=DEVICE, batch_size=RTDETR_BATCH, limit=None, imgsz=640)
            result['regime'] = regime
            result['protocol'] = 'test_operational_min8_v5_full_consensus_assisted/test (1871 images, 2467 boxes; model-assisted)'
            result_file.write_text(json.dumps(result, indent=2), encoding='utf-8')
            print(f'[SAVED] {result_file.name} | mAP50={result["accuracy"]["mAP50"]:.4f}')
        all_results[(regime, model)] = result
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
# 8. Aggregate matched overall and per-class CSVs.
overall_rows, class_rows = [], []
for (regime, model), result in all_results.items():
    a = result['accuracy']
    overall_rows.append({
        'Train': regime, 'Model': model, 'Precision': a['precision'], 'Recall': a['recall'],
        'F1': a['f1_score'], 'mAP50': a['mAP50'], 'mAP50:95': a['mAP50_95'],
        'MEAN_5': a['MEAN_5_defect_ap50'],
    })
    row = {'Train':regime, 'Model':model}
    for cls in CLASSES:
        row[f'{cls}_AP50'] = a['per_class'][cls]['ap50']
        row[f'{cls}_AP50_95'] = a['per_class'][cls]['ap50_95']
    class_rows.append(row)

overall = pd.DataFrame(overall_rows)
per_class = pd.DataFrame(class_rows)
overall.to_csv(OUTPUT_DIR/'benchmark_summary.csv', index=False)
per_class.to_csv(OUTPUT_DIR/'per_class_summary.csv', index=False)
display(overall)
print('Saved:', OUTPUT_DIR/'benchmark_summary.csv')
print('Saved:', OUTPUT_DIR/'per_class_summary.csv')


In [ ]:
# 9. One figure: Raw and Curated per-class performance on the SAME operational min-8 audit V5 test.
import matplotlib.pyplot as plt
import numpy as np

labels = {'yolo11n':'YOLO11n','yolo11m':'YOLO11m','yolo26n':'YOLO26n','yolo26m':'YOLO26m',
          'rtdetr50':'RT-DETR-R50','rtdetr101':'RT-DETR-R101'}
class_labels = ['Insulator','Broken insulator','Flashover','Bird nest','Foreign object','Broken strand']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, (suffix, title) in enumerate((('AP50','AP@0.5 (%)'),('AP50_95','AP@0.5:0.95 (%)'))):
    mats = {}
    for regime in ('Raw','Curated'):
        d = per_class[per_class.Train == regime].set_index('Model').reindex(MODEL_ORDER)
        mats[regime] = d[[f'{c}_{suffix}' for c in CLASSES]].to_numpy(float) * 100
    lo, hi = min(x.min() for x in mats.values()), max(x.max() for x in mats.values())
    for row, regime in enumerate(('Raw','Curated')):
        ax = axes[row, col]
        m = mats[regime]
        im = ax.imshow(m, cmap='Blues', vmin=lo, vmax=hi, aspect='auto')
        for i in range(m.shape[0]):
            for j in range(m.shape[1]):
                ax.text(j, i, f'{m[i,j]:.1f}', ha='center', va='center',
                        color='white' if im.norm(m[i,j]) > .58 else 'black', fontsize=10)
        ax.set_xticks(range(6), class_labels, rotation=30, ha='right')
        ax.set_yticks(range(6), [labels[m] for m in MODEL_ORDER])
        ax.set_title(f'{regime} training — {title}', fontweight='bold')
        fig.colorbar(im, ax=ax, fraction=.035, pad=.02)

fig.suptitle('Per-class outcomes on operational min-8 audit V5 (1,871 images, 2,467 boxes)', fontweight='bold')
fig.tight_layout()
png = OUTPUT_DIR/'per_class_raw_curated_operational_min8_v5.png'
pdf = OUTPUT_DIR/'per_class_raw_curated_operational_min8_v5.pdf'
fig.savefig(png, dpi=300, bbox_inches='tight')
fig.savefig(pdf, bbox_inches='tight')
plt.show()
print('Saved:', png)
print('Saved:', pdf)


## Reporting rule

Only the CSV and figure produced by this notebook may be described as a matched Raw-versus-Curated comparison on the human-reviewed test. Do not substitute validation results or operational-v5 exploratory results.
